# 02 — Data Cleaning & ETL Pipeline

This notebook contains the full Python ETL pipeline for the Lending Club dataset.

Every transformation step is logged with before/after statistics to ensure reproducibility.

**Input:** Raw gzip dataset (loaded via `01_data_loading.ipynb`)  
**Output:** `data/processed/lending_club_cleaned.csv`

In [ ]:
# ── Imports & load ────────────────────────────────────────────────
# Run 01_data_loading.ipynb first to populate `df` and `file_path`
import pandas as pd
import numpy as np
import re
import gzip
import os

# Reload from raw source (so this notebook is self-contained)
import kagglehub
path = kagglehub.dataset_download("ethon0426/lending-club-20072020q1")

data_file = None
for item in os.listdir(path):
    if item.endswith((".gzip", ".csv")):
        data_file = os.path.join(path, item)
        break

final_cols = [
    'loan_amnt', 'funded_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade',
    'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d',
    'loan_status', 'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line',
    'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
    'initial_list_status', 'collections_12_mths_ex_med', 'application_type', 'acc_now_delinq',
    'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim'
]

try:
    chunks = pd.read_csv(data_file, usecols=lambda c: c in final_cols,
                         compression='gzip', chunksize=100_000, low_memory=False)
except (gzip.BadGzipFile, pd.errors.ParserError):
    chunks = pd.read_csv(data_file, usecols=lambda c: c in final_cols,
                         chunksize=100_000, low_memory=False)

df = pd.concat(chunks, ignore_index=True)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")    

## Step 1 — Binary Encode Target Variable

Map `loan_status` → `loan_default` (1 = default, 0 = no default).

Statuses classified as default: *Charged Off, Default, Late (31–120 days), Late (16–30 days), Does not meet credit policy: Charged Off*

In [ ]:
# ── Step 1: Binary encode loan_status → loan_default ─────────────
default_flags = [
    'Charged Off', 'Default',
    'Late (31-120 days)', 'Late (16-30 days)',
    'Does not meet the credit policy. Status:Charged Off'
]

df['loan_default'] = df['loan_status'].apply(
    lambda x: 1 if x in default_flags else 0
)

print("loan_default distribution:")
print(df['loan_default'].value_counts(normalize=True).round(3))
print(f"\nOverall default rate: {df['loan_default'].mean()*100:.2f}%")    

## Step 2 — Fix Date Columns

Convert `issue_d` and `earliest_cr_line` from string (e.g. `Jan-2015`) to datetime.
Derive: `issue_year`, `issue_month`, `credit_age_months`.

In [ ]:
# ── Step 2: Parse date columns ────────────────────────────────────
df['issue_d']          = pd.to_datetime(df['issue_d'],          format='%b-%Y', errors='coerce')
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y', errors='coerce')

df['issue_year']  = df['issue_d'].dt.year
df['issue_month'] = df['issue_d'].dt.month
df['credit_age_months'] = (
    (df['issue_d'].dt.year  - df['earliest_cr_line'].dt.year) * 12 +
    (df['issue_d'].dt.month - df['earliest_cr_line'].dt.month)
).round(0)

print("issue_year range:", df['issue_year'].min(), "–", df['issue_year'].max())
print("credit_age_months stats:")
print(df['credit_age_months'].describe())    

## Step 3 — Parse String Columns

Convert `emp_length`, `term`, `int_rate`, and `revol_util` from string to numeric.

In [ ]:
# ── Step 3a: emp_length → numeric years ───────────────────────────
def parse_emp(val):
    if pd.isna(val): return np.nan
    if '10+' in str(val): return 10
    if '< 1'  in str(val): return 0
    nums = re.findall(r'\d+', str(val))
    return int(nums[0]) if nums else np.nan

df['emp_length_n'] = df['emp_length'].apply(parse_emp)
print("emp_length_n:", df['emp_length_n'].value_counts().sort_index().to_dict())

# ── Step 3b: term → numeric months ────────────────────────────────
df['term_n'] = df['term'].str.extract(r'(\d+)').astype(float)
print("\nterm_n unique values:", df['term_n'].unique())

# ── Step 3c: int_rate & revol_util → float (strip %) ──────────────
for col in ['int_rate', 'revol_util']:
    if df[col].dtype == object:
        df[col] = df[col].str.replace('%', '').str.strip().astype(float)
        print(f"{col} converted — range: {df[col].min():.1f}% – {df[col].max():.1f}%")    

## Step 4 — Handle Missing Values

- **Numerical:** median imputation
- **Categorical:** mode or `'Unknown'`

In [ ]:
# ── Step 4: Impute missing values ─────────────────────────────────
num_fill = ['emp_length_n', 'dti', 'revol_util', 'tot_cur_bal',
            'tot_hi_cred_lim', 'mort_acc', 'pub_rec_bankruptcies', 'credit_age_months']

missing_num_cols = []
for col in num_fill:
    if col not in df.columns:
        missing_num_cols.append(col)
        continue
    median_val    = df[col].median()
    missing_before = df[col].isnull().sum()
    df[col] = df[col].fillna(median_val)
    print(f"  {col}: filled {missing_before:,} nulls with median={median_val:.2f}")
if missing_num_cols:
    print("  Skipped (not in df):", ", ".join(missing_num_cols))

cat_fill = ['emp_length', 'home_ownership', 'verification_status']
for col in cat_fill:
    if col not in df.columns: continue
    df[col] = df[col].fillna('Unknown')    

## Step 5 — Cap Outliers (IQR Method)

Apply 3×IQR capping to `annual_inc`, `dti`, `revol_bal`, `tot_cur_bal` to reduce the effect of extreme values.

In [ ]:
# ── Step 5: Cap outliers ──────────────────────────────────────────
def cap_outliers(series, factor=3):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - factor * iqr, q3 + factor * iqr
    return series.clip(lower, upper)

for col in ['annual_inc', 'dti', 'revol_bal', 'tot_cur_bal']:
    before_max = df[col].max()
    df[col] = cap_outliers(df[col])
    print(f"  {col}: max {before_max:,.0f} → {df[col].max():,.0f}")    

## Step 6 — Feature Engineering (KPI Columns)

Derive additional columns used in EDA, statistical analysis, and Tableau.

In [ ]:
# ── Step 6: Feature engineering ───────────────────────────────────
df['loan_to_income'] = (df['loan_amnt'] / df['annual_inc'].replace(0, np.nan)).round(4)

if 'fico_range_low' in df.columns and 'fico_range_high' in df.columns:
    df['fico_avg'] = (df['fico_range_low'] + df['fico_range_high']) / 2
else:
    df['fico_avg'] = np.nan
    print("  fico_avg: not available in this column selection — set to NaN")

df['delinq_flag']    = (df['delinq_2yrs'] > 0).astype(int)
df['high_risk_flag'] = (df['dti'] > 30).astype(int)

print(f"\nFinal shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(df[['loan_to_income', 'fico_avg', 'delinq_flag', 'high_risk_flag']].describe())    

## Step 7 — Save Cleaned Dataset

In [ ]:
# ── Step 7: Save to data/processed/ ──────────────────────────────
import os
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)

df.to_csv(f'{output_dir}/lending_club_cleaned.csv', index=False)
print(f"\n✅ Saved: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"   → {output_dir}/lending_club_cleaned.csv")    